# AlgorimSeek – Inference & Fine-Tuning Test

Load the merged AlgorimSeek model and run tests: code generation, `/compile` command, and vision (if image available).

In [ ]:
# ── Imports & Setup ─────────────────────────────────────────────
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
import warnings
warnings.filterwarnings('ignore')

# Path to the merged model (adjust if needed)
merged_path = "./algorimseek_output/merged"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
# ── Load Model & Processor ───────────────────────────────────────
print("Loading model...")
test_model = Qwen2VLForConditionalGeneration.from_pretrained(
    merged_path,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)
test_processor = AutoProcessor.from_pretrained(merged_path)
print("Model loaded successfully.")

In [ ]:
# ── System Prompt & Inference Function ──────────────────────────
SYSTEM_PROMPT = """You are AlgorimSeek, an expert AI system for the Algorim programming language (algo a47).

## Algorim Language Rules:
- Functions declared with: `Action name(params): return_type`
- Variable section: `var` ... `begin`
- Assignment operator: `<---`
- Loops: `for (i <--- 0 to n) do` ... `endfor` | `while (cond) do` ... `done`
- Conditionals: `if (cond) then` ... `else` ... `endif`
- Comments: `// text` | Arabic comments supported
- Types: int, float, bool, char, arr, array, string, node
- Return: `action_name <--- value` (assign to function name)

## Supported Commands:
- `/imagine [description]` → Generate code visualization image
- `/compile [code]` → Compile to bytecode (AlgorimVM)
- `/debug [code]` → Step-by-step execution trace
- `/execute [code]` → Run code and show output
- `/explain [code]` → Detailed explanation

## Capabilities:
- Generate Algorim code from text descriptions
- Read code from images (vision)
- Long-context reasoning for complex algorithms
- Bilingual: Arabic and English
- Think step-by-step inside <thinking>...</thinking> tags"""

def ask_algorimseek(prompt, image_path=None, max_tokens=1024):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': []}
    ]
    if image_path:
        msgs[1]['content'].append({'type': 'image', 'image': image_path})
    msgs[1]['content'].append({'type': 'text', 'text': prompt})
    
    text = test_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = test_processor(text=[text], return_tensors='pt').to(DEVICE)
    
    with torch.no_grad():
        out = test_model.generate(**inputs, max_new_tokens=max_tokens,
                                   temperature=0.7, top_p=0.9, do_sample=True)
    return test_processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
# ── Test 1: Text → Algorim Code ─────────────────────────────────
print("=" * 60)
print("TEST 1: Code Generation")
print("=" * 60)
r1 = ask_algorimseek("اكتب دالة Algorim لحساب ارتفاع شجرة ثنائية")
print(r1)

In [ ]:
# ── Test 2: /compile Command ─────────────────────────────────────
print("\n" + "=" * 60)
print("TEST 2: /compile command")
print("=" * 60)
r2 = ask_algorimseek("""/compile
Action binary_search(T: arr, n: int, target: int): int
var
    low: int
    high: int
    mid: int
begin
    low  <--- 0
    high <--- n - 1
    binary_search <--- -1
    while (low <= high) do
        mid <--- (low + high) div 2
        if (T[mid] = target) then
            binary_search <--- mid
            low <--- high + 1
        else
            if (T[mid] < target) then
                low <--- mid + 1
            else
                high <--- mid - 1
            endif
        endif
    done
end""")
print(r2)

In [ ]:
# ── Test 3: Vision → Code (optional) ─────────────────────────────
print("\n" + "=" * 60)
print("TEST 3: Image → Code (Vision)")
print("=" * 60)
image_path = "images_generated/demo_render.png"
try:
    r3 = ask_algorimseek("اقرأ هذه الصورة واشرح الكود", image_path=image_path)
    print(r3)
except FileNotFoundError:
    print(f"Image not found at {image_path}. Skipping vision test.")

print("\n✅ All inference tests passed!")